# ADVC — Kaggle Notebook (Tiny-ImageNet, T4 GPU)

**Environment:** Kaggle Notebooks · GPU T4 x2 (16 GB each) · ~4 CPU cores · ~30 GB RAM

Run cells top-to-bottom. Every phase is **resumable** — re-running a cell skips already-completed rows.

### Before running — notebook Settings (right sidebar)
1. **Accelerator** → `GPU T4 x2`
2. **Internet** → `On` (needed for `git clone`, `pip install`, HF model download)
3. **Add-ons → Secrets** → add `HF_TOKEN` = your HuggingFace token (https://huggingface.co/settings/tokens)
4. **Add-ons → Datasets** → attach a Tiny-ImageNet dataset (e.g. search `tiny-imagenet` — the standard Stanford CS231n `tiny-imagenet-200.zip`)

> **To keep it running after you shut down your PC**, see the final markdown cell **"Running headless — Save & Run All (Commit)"**. The short version: use **Save Version → Save & Run All (Commit)**, which runs the whole notebook on Kaggle's servers with your machine off.

## Cell 1 — Verify GPU
Confirm a T4 is attached before starting long experiments.

In [ ]:
# Cell 1 — Verify GPU
import torch

print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    free, total = torch.cuda.mem_get_info(0)
    print('GPU name       :', name)
    print(f'Free VRAM      : {free/1e9:.1f} GB / {total/1e9:.1f} GB total')
    if 'T4' not in name:
        print('\nWARNING: expected Tesla T4 — check Settings > Accelerator > GPU T4 x2')
    else:
        print('\nT4 confirmed. Ready to proceed.')
else:
    print('\nWARNING: no GPU. Set Settings > Accelerator > GPU T4 x2, then restart.')

## Cell 2 — Set HF token + install dependencies

`transformers` is pinned `<5.0` (v5 removed `ViTForImageClassification` from the top-level registry, which breaks bitsandbytes INT8/INT4). Kaggle's pre-installed torch already sees the T4, so we don't reinstall it.

In [ ]:
# Cell 2 — HF token + dependencies
import os, subprocess, sys

# Pull HF_TOKEN from Kaggle Secrets (set it in Add-ons > Secrets)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle Secrets.')
except Exception as e:
    print('WARNING: could not load HF_TOKEN secret:', e)
    print('Add it in Settings > Add-ons > Secrets, or downloads may be rate-limited.')

packages = ['timm', 'torchattacks', 'bitsandbytes', 'optimum', 'pyyaml',
            'tqdm', 'accelerate', 'huggingface_hub', 'transformers>=4.44.0,<5.0']
res = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages,
                     capture_output=True, text=True)
if res.returncode != 0:
    print('pip error:\n', res.stderr[-2000:])
else:
    print('Installed:', ', '.join(packages))

## Cell 3 — Clone repo and set working directory

Clones the code into `/kaggle/working/ADVC` and `cd`s into it. Re-runnable: if the repo is already there it just pulls the latest.

In [ ]:
# Cell 3 — Clone / update repo, set cwd
import os, sys, subprocess

REPO_URL = 'https://github.com/Jmanav/ADVC.git'
REPO_DIR = '/kaggle/working/ADVC'

if not os.path.isdir(REPO_DIR):
    print('Cloning', REPO_URL)
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print('Repo exists — pulling latest')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('cwd is now:', os.getcwd())
assert os.path.exists('configs/base.yaml'), 'configs/base.yaml missing — clone failed?'

## Cell 4 — Locate & extract Tiny-ImageNet, then prepare ImageFolder dirs

Finds the attached Tiny-ImageNet dataset under `/kaggle/input/`, extracts (if it's a zip) into `/kaggle/working/`, then runs `scripts/prepare_tiny_imagenet.py` to build the `train_if/` and `val_if/` folders the loaders expect. Idempotent — safe to re-run.

In [ ]:
# Cell 4 — Extract + prepare Tiny-ImageNet
import os, glob, zipfile, subprocess, sys
from pathlib import Path

WORK_ROOT = '/kaggle/working/tiny-imagenet-200'

if not os.path.isdir(WORK_ROOT):
    # 1) Already-extracted folder somewhere under /kaggle/input?
    found_dir = None
    for p in glob.glob('/kaggle/input/**/tiny-imagenet-200', recursive=True):
        if os.path.isdir(p):
            found_dir = p
            break
    if found_dir:
        print('Found extracted dataset at', found_dir, '— symlinking')
        os.symlink(found_dir, WORK_ROOT)
    else:
        # 2) Look for a zip to extract
        zips = glob.glob('/kaggle/input/**/tiny-imagenet-200.zip', recursive=True)
        if not zips:
            zips = glob.glob('/kaggle/input/**/*tiny*imagenet*.zip', recursive=True)
        assert zips, ('No tiny-imagenet-200 folder or zip found under /kaggle/input. '
                      'Attach a Tiny-ImageNet dataset in Settings > Add-ons > Datasets.')
        print('Extracting', zips[0])
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall('/kaggle/working/')
        assert os.path.isdir(WORK_ROOT), 'Extraction did not produce tiny-imagenet-200/'
else:
    print('Tiny-ImageNet already present at', WORK_ROOT)

# Build ImageFolder-ready train_if/ and val_if/
print('\nPreparing ImageFolder layout ...')
res = subprocess.run([sys.executable, 'scripts/prepare_tiny_imagenet.py', '--root', WORK_ROOT],
                     capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print('prepare failed:\n', res.stderr[-2000:])
else:
    for d in ['train_if', 'val_if']:
        path = os.path.join(WORK_ROOT, d)
        n = len(os.listdir(path)) if os.path.isdir(path) else 0
        print(f'  {d}: {n} class folders')
    print('Tiny-ImageNet ready.')

## Cell 5 — Create output directories

Results and checkpoints land in `/kaggle/working/ADVC/results/`. Everything under `/kaggle/working/` is captured as notebook output when you commit.

In [ ]:
# Cell 5 — Output directories
import os
for d in ['results', 'results/checkpoints/at', 'results/checkpoints/atkd', 'results/figures']:
    os.makedirs(d, exist_ok=True)
    print('Ready:', d)

## Cell 5b — (Optional) Restore previous results

If a **previous commit** saved results and you attached that notebook's output as a dataset (or you're re-running the same notebook, in which case `/kaggle/working` persists between commits of the *same* notebook), copy old CSVs/checkpoints back in so the resumability logic skips finished work. Edit `PREV` to your attached results dataset path, or leave as-is to skip.

In [ ]:
# Cell 5b — Restore prior results (optional)
import os, glob, shutil

PREV = ''  # e.g. '/kaggle/input/advc-results'  (leave '' to skip)

if PREV and os.path.isdir(PREV):
    os.makedirs('results', exist_ok=True)
    for csv in glob.glob(os.path.join(PREV, '*.csv')):
        shutil.copy(csv, os.path.join('results', os.path.basename(csv)))
        print('Restored', os.path.basename(csv))
    for sub in ['at', 'atkd']:
        src = os.path.join(PREV, 'checkpoints', sub)
        if os.path.isdir(src):
            shutil.copytree(src, f'results/checkpoints/{sub}', dirs_exist_ok=True)
            print('Restored checkpoints/', sub)
else:
    print('No PREV set — starting fresh (or relying on same-notebook /kaggle/working persistence).')

## Cell 6 — Smoke test: load FP32 DeiT-S
One forward pass to confirm the environment works before long runs.

In [ ]:
# Cell 6 — Smoke test
import torch
from models.loader import load_config, load_model

cfg = load_config('configs/base.yaml')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dataset in config:', cfg['dataset']['name'])

model = load_model('deit_small', 'fp32', cfg, device=device)
model.eval()
dummy = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    out = model(dummy)
    if hasattr(out, 'logits'):
        out = out.logits
print('Output shape:', tuple(out.shape), '(expected (2, 1000))')

del model, dummy
torch.cuda.empty_cache()
print('Smoke test passed.')

## Cell 7 — Phase 1: no-defense baseline
Sweeps fp32/int8/int4 × FGSM/PGD/Patch → `results/phase1_results.csv`. Fully resumable.

In [ ]:
# Cell 7 — Phase 1
import subprocess, sys, os

proc = subprocess.Popen([sys.executable, 'experiments/eval_phase1.py', '--model', 'deit_small'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                        bufsize=1, env={**os.environ})
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('\n[Phase 1] exit code', proc.returncode)

## Cell 8 — Phase 2a: Adversarial Training (AT)
Fine-tunes each compression level (7 epochs), then evaluates 3 attacks → `results/phase2_at_results.csv`. Checkpoints saved every epoch to `results/checkpoints/at/`.

In [ ]:
# Cell 8 — Phase 2a (AT)
import subprocess, sys, os

for compression in ['fp32', 'int8', 'int4']:
    print('=' * 60)
    print('Phase 2a: AT —', compression)
    print('=' * 60)
    proc = subprocess.Popen([sys.executable, 'experiments/eval_phase2_at.py',
                             '--compression', compression],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            bufsize=1, env={**os.environ})
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    print(f'[Phase 2a {compression}] exit code', proc.returncode, '\n')

## Cell 9 — Phase 2b: AT + Knowledge Distillation (AT+KD)
Same as 2a plus a frozen FP32 teacher (KL supervision) → `results/phase2_atkd_results.csv`. Checkpoints in `results/checkpoints/atkd/`.

In [ ]:
# Cell 9 — Phase 2b (AT+KD)
import subprocess, sys, os

for compression in ['fp32', 'int8', 'int4']:
    print('=' * 60)
    print('Phase 2b: AT+KD —', compression)
    print('=' * 60)
    proc = subprocess.Popen([sys.executable, 'experiments/eval_phase2_atkd.py',
                             '--compression', compression],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            bufsize=1, env={**os.environ})
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    print(f'[Phase 2b {compression}] exit code', proc.returncode, '\n')

## Cell 10 — Phase 3: combined attack vs all defenses
Combined FGSM→PGD→Patch against none/AT/AT+KD → `results/phase3_results.csv`. Requires Phase 2 checkpoints to exist.

In [ ]:
# Cell 10 — Phase 3
import subprocess, sys, os

proc = subprocess.Popen([sys.executable, 'experiments/eval_phase3.py', '--model', 'deit_small'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                        bufsize=1, env={**os.environ})
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('\n[Phase 3] exit code', proc.returncode)

## Cell 11 — Preview all results
Load every completed CSV. Run at any point to check progress.

In [ ]:
# Cell 11 — Preview results
import pandas as pd, os

report = {
    'Phase 1 — No Defense':     'results/phase1_results.csv',
    'Phase 2a — AT':            'results/phase2_at_results.csv',
    'Phase 2b — AT+KD':         'results/phase2_atkd_results.csv',
    'Phase 3 — Combined':       'results/phase3_results.csv',
}
cols = ['compression', 'defense', 'attack', 'clean_acc', 'robust_acc', 'asr', 'robustness_gap']
for title, path in report.items():
    print('\n' + '=' * 60 + '\n' + title + '\n' + '=' * 60)
    if not os.path.exists(path):
        print('  not yet generated:', path); continue
    df = pd.read_csv(path)
    if df.empty:
        print('  file exists but empty'); continue
    print(df[[c for c in cols if c in df.columns]].to_string(index=False))
    print(f'  {len(df)} row(s)')

## Cell 12 — (Optional) Generate paper figures
Run after the phase CSVs exist. Outputs to `results/figures/`.

In [ ]:
# Cell 12 — Paper figures
import subprocess, sys, os

proc = subprocess.Popen([sys.executable, 'utils/paper_figures.py', '--n-samples', '4', '--n-eval', '200'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                        bufsize=1, env={**os.environ})
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('\n[Figures] exit code', proc.returncode)
figs = os.path.join('results', 'figures')
if os.path.isdir(figs):
    for f in sorted(os.listdir(figs)):
        if f != '.gitkeep':
            print(' ', f, f'{os.path.getsize(os.path.join(figs, f))/1024:.0f} KB')

---
## Running headless — keep it running with your PC off

Kaggle runs notebooks **on its own servers**, not your machine — so you can close the browser and shut down your PC. There are two modes:

### A. Interactive session (what you get by pressing ▶ on cells)
- The kernel keeps running server-side **only while the session is alive**. If you close the tab it stays up for a while, but an idle interactive session is eventually reclaimed (roughly ~20–40 min idle, ~9–12 h max). **Not reliable for an overnight, PC-off run.** Don't rely on this for the full pipeline.

### B. Save & Run All (Commit) — use this to survive a PC shutdown ✅
1. Top-right → **Save Version**.
2. Choose **Save & Run All (Commit)** (not "Quick Save").
3. Confirm **GPU T4 x2** and **Internet On** are still selected, plus the `HF_TOKEN` secret and Tiny-ImageNet dataset are attached.
4. Click **Save**. Kaggle spins up a fresh headless machine, runs **every cell top-to-bottom**, and **you can now close the browser and shut down your PC.** It keeps running on Kaggle's servers.
5. Track progress under the notebook's **Versions** tab (or *Your Work → Notebooks*). When it finishes you'll get a completed version whose **Output** tab holds everything written to `/kaggle/working/` — your `results/*.csv`, checkpoints, and figures, all downloadable.

**Committed-run limits:** up to **12 h** wall-clock per run and **30 GPU h/week** on the free tier. The full 36-row matrix + figures may exceed one 12 h window — that's fine because every phase is resumable:

### Chaining commits past the 12 h limit
- Within a run, if the 12 h cap hits mid-phase, the CSV rows already written are preserved in that version's output.
- To continue: attach the finished version's **Output** as an input dataset, set `PREV` in **Cell 5b** to that path (e.g. `/kaggle/input/<your-notebook>/results`), and **Save & Run All** again. Resumability skips completed rows and picks up where it stopped.
- Repeat until Phase 3 + figures are done.

### Tips
- **Don't** edit the notebook while a commit is running — it runs the *saved* version, so editing does nothing until the next commit.
- Watch weekly GPU quota: *Settings → the accelerator usage meter*. AT/AT+KD training dominates it.
- If you only want a subset (e.g. Phase 1 to validate the pipeline first), delete or comment out the later phase cells before committing so you don't burn GPU hours.